# StatParse - Pipeline d'Analyse Documentaire
## Evaluation sur OmniDocBench - ENSAE 2025 - Projet RAG

---
Ce notebook presente les resultats de notre pipeline **StatParse** evalue sur le benchmark [OmniDocBench](https://github.com/opendatalab/OmniDocBench), compare au baseline **Docling**.

> **Les figures ne sont plus construites dans ce notebook.**
> `tools/generate_figures.py` les reconstruit toutes les huit directement
> depuis `result/*_quick_match_metric_result.json`. Aucune valeur n'y est
> ecrite en dur, donc les figures ne peuvent pas diverger des scores.
> La derniere cellule appelle ce script et affiche le resultat.

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# ── Style global ──────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.titleweight': 'bold',
})

# ── Couleurs ──────────────────────────────────────────────────────
C_STATPARSE = '#2563EB'   # bleu vif
C_DOCLING   = '#16A34A'   # vert
C_LIGHT_SP  = '#DBEAFE'
C_LIGHT_DL  = '#DCFCE7'

print('✅ Imports OK')

✅ Imports OK


In [2]:
# ── Chargement des résultats ──────────────────────────────────────
RESULT_DIR = '../result'

with open(f'{RESULT_DIR}/statparse_quick_match_metric_result.json') as f:
    sp = json.load(f)

with open(f'{RESULT_DIR}/docling_quick_match_metric_result.json') as f:
    dl = json.load(f)

print('✅ Résultats chargés')
print(f'   StatParse — catégories: {list(sp.keys())}')
print(f'   Docling   — catégories: {list(dl.keys())}')

✅ Résultats chargés
   StatParse — catégories: ['text_block', 'display_formula', 'table', 'reading_order']
   Docling   — catégories: ['text_block', 'display_formula', 'table', 'reading_order']


In [3]:
# ── Tableau de synthèse ───────────────────────────────────────────
def extract_scores(result):
    """Extrait les métriques principales d'un résultat JSON."""
    scores = {}
    scores['text_Edit_dist']      = result['text_block']['all']['Edit_dist']['ALL_page_avg']
    scores['formula_Edit_dist']   = result['display_formula']['all']['Edit_dist']['ALL_page_avg']
    scores['table_TEDS']          = result['table']['page']['TEDS']['ALL'] * 100
    scores['table_TEDS_struct']   = result['table']['page']['TEDS_structure_only']['ALL'] * 100
    scores['table_Edit_dist']     = result['table']['all']['Edit_dist']['ALL_page_avg']
    scores['reading_order']       = result['reading_order']['all']['Edit_dist']['ALL_page_avg']
    # Score global (convention OmniDocBench)
    scores['overall'] = (
        (1 - scores['text_Edit_dist']) * 100 +
        (1 - scores['reading_order']) * 100 +
        scores['table_TEDS']
    ) / 3
    return scores

sp_scores = extract_scores(sp)
dl_scores = extract_scores(dl)

df = pd.DataFrame({
    'StatParse': sp_scores,
    'Docling':   dl_scores
}).T.round(3)

# Affichage lisible
df_display = df.copy()
df_display.columns = [
    'Text Edit↓', 'Formula Edit↓', 
    'Table TEDS↑', 'Table TEDS-S↑',
    'Table Edit↓', 'Reading Order↓', 'Overall↑'
]

print('\n📊 SCORES GLOBAUX\n')
print(df_display.to_string())
df_display


📊 SCORES GLOBAUX

           Text Edit↓  Formula Edit↓  Table TEDS↑  Table TEDS-S↑  Table Edit↓  Reading Order↓  Overall↑
StatParse       0.384          0.960       15.574         17.433        0.834           0.353    47.316
Docling         0.176          0.345       66.734         73.664        0.643           0.183    76.923


,Text Edit↓,Formula Edit↓,Table TEDS↑,Table TEDS-S↑,Table Edit↓,Reading Order↓,Overall↑
StatParse,0.384,0.960,15.574,17.433,0.834,0.353,47.316
Docling,0.176,0.345,66.734,73.664,0.643,0.183,76.923


In [ ]:
# ── Generation des figures ─────────────────────────────────────────
# Delegue a tools/generate_figures.py : source unique de verite pour les
# huit figures, lues directement depuis result/*.json.
import subprocess
import sys
from pathlib import Path

here = Path.cwd()
candidates = [
    here / "generate_figures.py",           # notebook lance depuis tools/
    here / "tools" / "generate_figures.py",  # ou depuis la racine du depot
]
script = next((p for p in candidates if p.exists()), None)
if script is None:
    raise FileNotFoundError(
        "generate_figures.py introuvable - lancer le notebook depuis "
        "tools/ ou depuis la racine du depot"
    )

run = subprocess.run([sys.executable, str(script)],
                     capture_output=True, text=True)
print(run.stdout or run.stderr)

# ── Affichage ──────────────────────────────────────────────────────
from IPython.display import Image, display

for png in sorted(script.parent.glob("fig*.png")):
    print(png.name)
    display(Image(filename=str(png)))
